# Semantic Kernel

在这个代码示例中，你将使用 [Semantic Kernel](https://aka.ms/ai-agents-beginners/semantic-kernel) AI 框架创建一个基本的智能体。

本示例的目标是向你展示我们稍后在实现不同智能体模式的附加代码示例中使用的步骤。

## 导入所需的 Python 包

In [1]:
import os 
from typing import Annotated
from openai import AsyncOpenAI

from dotenv import load_dotenv

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.functions import kernel_function

## 创建客户端

在本示例中，我们将使用 [GitHub Models](https://aka.ms/ai-agents-beginners/github-models) 来访问 LLM。

`ai_model_id` 被定义为 `gpt-4o-mini`。尝试将模型更改为 GitHub Models 市场上可用的其他模型，以查看不同的结果。

为了使用用于 GitHub Models 的 `base_url` 的 `Azure Inference SDK`，我们将在 Semantic Kernel 中使用 `OpenAIChatCompletion` 连接器。Semantic Kernel 还提供了其他 [可用的连接器](https://learn.microsoft.com/semantic-kernel/concepts/ai-services/chat-completion)，可用于其他模型提供商。

In [2]:
import random   

# Define a sample plugin for the sample

class DestinationsPlugin:
    """A List of Random Destinations for a vacation."""

    def __init__(self):
        # List of vacation destinations
        self.destinations = [
            "Barcelona, Spain",
            "Paris, France",
            "Berlin, Germany",
            "Tokyo, Japan",
            "Sydney, Australia",
            "New York, USA",
            "Cairo, Egypt",
            "Cape Town, South Africa",
            "Rio de Janeiro, Brazil",
            "Bali, Indonesia"
        ]
        # Track last destination to avoid repeats
        self.last_destination = None

    @kernel_function(description="Provides a random vacation destination.")
    def get_random_destination(self) -> Annotated[str, "Returns a random vacation destination."]:
        # Get available destinations (excluding last one if possible)
        available_destinations = self.destinations.copy()
        if self.last_destination and len(available_destinations) > 1:
            available_destinations.remove(self.last_destination)

        # Select a random destination
        destination = random.choice(available_destinations)

        # Update the last destination
        self.last_destination = destination

        return destination

In [4]:
load_dotenv()
client = AsyncOpenAI(
    api_key=os.environ.get("GITHUB_TOKEN"), 
    # base_url="https://models.inference.ai.azure.com/",
    base_url=os.environ.get("API_URL"),
)

# Create an AI Service that will be used by the `ChatCompletionAgent`
chat_completion_service = OpenAIChatCompletion(
    # ai_model_id="gpt-4o-mini",
    ai_model_id=os.environ.get("MODEL_FREE_8B"),
    async_client=client,
)

## 创建智能体

下面我们创建一个名为 `TravelAgent` 的智能体。

在这个示例中，我们使用非常简单的指令。你可以更改这些指令，看看智能体如何以不同的方式响应。

In [5]:
agent = ChatCompletionAgent(
    service=chat_completion_service, 
    plugins=[DestinationsPlugin()],
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
)

## 运行智能体

现在我们可以通过定义一个 `ChatHistoryAgentThread` 类型的线程来运行智能体。任何所需的系统消息都通过智能体的 `invoke_stream` 方法的 `messages` 关键字参数提供。

定义这些后，我们创建一个 `user_inputs`，即用户发送给智能体的内容。在这种情况下，我们将这条消息设置为 `Plan me a sunny vacation`。

随时更改此消息，看看智能体如何以不同的方式响应。

In [6]:
async def main():
    # Create a new thread for the agent
    # If no thread is provided, a new thread will be
    # created and returned with the initial response
    thread: ChatHistoryAgentThread | None = None

    user_inputs = [
        "Plan me a day trip.",
    ]

    for user_input in user_inputs:
        print(f"# User: {user_input}")
        first_chunk = True
        async for response in agent.invoke_stream(
            messages=user_input, thread=thread,
        ):
            # 5. Print the response
            if first_chunk:
                print(f"# {response.name}: ", end="", flush=True)
                first_chunk = False
            print(f"{response}", end="", flush=True)
            thread = response.thread
        print()

    # Clean up the thread
    await thread.delete() if thread else None

await main()

# User: Plan me a day trip.
# TravelAgent: Here's a day trip plan for Tokyo, Japan:

**Morning:**
- Start your day with a visit to **Senso-ji Temple**, the oldest temple in Tokyo and a great place to experience traditional Japanese culture.
- Walk through the **Asakusa** district, known for its traditional atmosphere and the iconic **Kaminarimon Gate**.

**Midday:**
- Head to **Ueno Park** for a relaxed lunch and some leisure time. The park has a large zoo, a botanical garden, and the **Ueno Art Museum**.
- Enjoy a local lunch at one of the many food stalls in **Ueno Park** or try a nearby restaurant for dishes like sushi or ramen.

**Afternoon:**
- Explore the **Meiji Shrine**, a serene and historic site nestled in the forest, offering a peaceful escape from the city's hustle.
- Visit the **Tokyo National Museum** to see a vast collection of Japanese art and artifacts.

**Evening:**
- End your day with a stroll through **Shibuya Crossing**, one of the world's busiest pedestrian crossi